In [ ]:
import pandas as pd
import os
from datetime import datetime

# Main configuration
DC_LIST = ['DC1', 'DC2', 'DC4', 'DC5']

DC_STOCK_SUFFIXES = ["DCStockQty", "DCStockValue", "DOHDC"]

DC_REPLACE_SUFFIXES = {
    "DCStockQty":   "Remain_StockQty",
    "DCStockValue": "Remain_StockValue",
}

DOH_CAP = 365

PATHS = {
    "input_folder":   os.getenv("CJ_STOCK_SOURCE_DIR", "data/raw/cj_stock"),
    "dc_stock_file":  os.getenv("CLEANED_DC_STOCK_PATH", "data/processed/cleaned_DC_daily_stock.xlsx"),
    "output_folder":  os.getenv("WEEKLY_STOCK_OUTPUT_DIR", "data/output/weekly"),
}

FILE_PREFIX    = "Sahamit Report "
FILE_EXTENSION = ".xlsx"
SHEET_NAME     = "Sahamit Report"
HEADER_ROW     = 2
# ============================================================

# HELPERS
def _build_dc_columns(dc_list, suffixes):
    """Generate column names like DC1_DCStockQty for all DCs."""
    return [f"{dc}_{suffix}" for dc in dc_list for suffix in suffixes]


def _build_replace_map(dc_list, suffix_map):
    """Build {main_col: merged_col} for all DCs.
    e.g. DC1_DCStockQty -> DC1_Remain_StockQty
    """
    return {
        f"{dc}_{main}": f"{dc}_{merged}"
        for dc in dc_list
        for main, merged in suffix_map.items()
    }


def get_latest_file(folder, prefix, extension):
    """Return the filename with the most recent date stamp, or None."""
    files = [
        f for f in os.listdir(folder)
        if f.startswith(prefix) and f.endswith(extension)
    ]
    dated = []
    for f in files:
        try:
            date_str = f.replace(prefix, "").replace(extension, "").strip()
            dated.append((f, datetime.strptime(date_str, "%d-%m-%Y")))
        except ValueError:
            continue
    return max(dated, key=lambda x: x[1])[0] if dated else None



# CORE PROCESSING
def replace_stock_from_dc(df, replace_map):
    """Overwrite main columns with merged DC daily stock where available."""
    for main_col, merged_col in replace_map.items():
        if merged_col in df.columns:
            df[main_col] = df[main_col].where(
                df[merged_col].isnull(), df[merged_col]
            )
    # drop the temporary merged columns
    cols_to_drop = [c for c in replace_map.values() if c in df.columns]
    return df.drop(columns=cols_to_drop)


def recalculate_doh(df, dc_list):
    """Recalculate Days-On-Hand for specified DCs."""
    for dc in dc_list:
        avg_col   = f"{dc}_AvgSaleQty90D"
        stock_col = f"{dc}_DCStockQty"
        doh_col   = f"{dc}_DOHDC"

        if avg_col not in df.columns or stock_col not in df.columns:
            continue

        df[avg_col] = df[avg_col].fillna(0)
        df[doh_col] = (
            df[stock_col]
            .div(df[avg_col])
            .replace([float("inf"), -float("inf")], DOH_CAP)
            .fillna(0)
        )
    return df


def clean_cj_stock():
    # 1. Find latest report
    latest = get_latest_file(PATHS["input_folder"], FILE_PREFIX, FILE_EXTENSION)
    if not latest:
        print("No valid files found.")
        return
    print(f"Loading: {latest}")

    # 2. Read CJ stock
    cj = pd.read_excel(
        os.path.join(PATHS["input_folder"], latest),
        sheet_name=SHEET_NAME, header=HEADER_ROW,
    )

    # Exclude division contains 'UNO'
    cj = cj[~cj["Division"].str.contains("UNO")]
    print("Excluded division containing 'UNO' from CJ stock data.")

    # 3. Fill NAs in DC columns
    fill_cols = _build_dc_columns(DC_LIST, DC_STOCK_SUFFIXES)
    existing = [c for c in fill_cols if c in cj.columns]
    cj[existing] = cj[existing].fillna(0)

    # 4. Merge DC daily stock
    dc_stock = pd.read_excel(
        PATHS["dc_stock_file"], sheet_name="Pivot_DC_stock"
    ).rename(columns={"CJ_Item": "Product"})
    print(f"Loading DC Stock: {PATHS['dc_stock_file']}")
    cj = cj.merge(dc_stock, on="Product", how="left")

    # 5. Replace & recalculate
    replace_map = _build_replace_map(DC_LIST, DC_REPLACE_SUFFIXES)
    cj = replace_stock_from_dc(cj, replace_map)
    print("Updated SHM DC stock with cleaned DC daily Stock.")
    cj = recalculate_doh(cj, DC_LIST)
    print("Recalculated DOH for DCs.")

    # 6. Save
    out = os.path.join(PATHS["output_folder"], latest)
    cj.to_excel(out, index=False)
    print(f"Saved: {out}")


if __name__ == "__main__":
    clean_cj_stock()